<a href="https://colab.research.google.com/github/Narendra725/Power_BI_Spark_Labs/blob/main/Power%20BI/Automations/Power%20Bi%20Desktop/Mach3/power_bi_objects_creation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [212]:
# import shutil
# import os

# # List of local items to exclude from deletion
# exclude = ['Power_BI_Spark_Labs', '.config', 'sample_data']

# print("Cleaning up local workspace...")
# for item in os.listdir('/content'):
#     item_path = os.path.join('/content', item)
#     if item not in exclude:
#         try:
#             if os.path.isfile(item_path) or os.path.islink(item_path):
#                 os.unlink(item_path)
#                 print(f"Deleted file: {item}")
#             elif os.path.isdir(item_path):
#                 shutil.rmtree(item_path)
#                 print(f"Deleted folder: {item}")
#         except Exception as e:
#             print(f"Failed to delete {item}: {e}")

# print("Cleanup complete. Repository and system configs preserved.")

Cleaning up local workspace...
Cleanup complete. Repository and system configs preserved.


In [213]:
import os

# Your Git Repo Details
REPO_URL = "https://github.com/Narendra725"
REPO_NAME = "Power_BI_Spark_Labs"

# Setting the working root to Mach3
REPO_DIR = "/content/Power_BI_Spark_Labs"
MACH3_ROOT = os.path.join(REPO_DIR, 'Power BI/Automations/Power Bi Desktop/Mach3')

print(f"Checking for repository at: {REPO_DIR}")
if not os.path.exists(REPO_DIR):
    print(f"Repository not found. Cloning {REPO_NAME}...")
    !git clone {REPO_URL}/{REPO_NAME}.git
    print(f"Successfully cloned {REPO_NAME}")
    !git pull
else:
    print(f"Repository already exists at {REPO_DIR}. Skipping clone.")

# Create Mach3 if it doesn't exist
os.makedirs(MACH3_ROOT, exist_ok=True)
print(f"Active Root: {MACH3_ROOT}")

Active Root: /content/Power_BI_Spark_Labs/Power BI/Automations/Power Bi Desktop/Mach3


### 1.1 Secure Git Authentication
To push changes, we need to authenticate.
1. Go to your GitHub Settings -> Developer Settings -> Personal Access Tokens -> Tokens (classic).
2. Generate a token with `repo` permissions.
3. Add it to Colab's Secrets (left sidebar 🔑) with the name `GITHUB_TOKEN`.

In [214]:
from google.colab import userdata
import os

# Configuration - Updated for your repository
USERNAME = "Narendra725"
REPO_NAME = "Power_BI_Spark_Labs"

try:
    token = userdata.get('GITHUB_TOKEN')
    # Re-construct the authenticated URL
    AUTH_REPO_URL = f"https://{token}@github.com/{USERNAME}/{REPO_NAME}.git"

    # Update the remote or clone if it doesn't exist
    if not os.path.exists(REPO_NAME):
        !git clone {AUTH_REPO_URL}
    else:
        %cd {REPO_NAME}
        !git remote set-url origin {AUTH_REPO_URL}
        !git pull origin main
        %cd ..
    print(f"Successfully synced: {REPO_NAME}")
except Exception as e:
    print(f"Setup Error: {e}")
    print("Please add 'GITHUB_TOKEN' to Colab Secrets (🔑) and toggle 'Notebook access'.")

/content/Power_BI_Spark_Labs
From https://github.com/Narendra725/Power_BI_Spark_Labs
 * branch            main       -> FETCH_HEAD
Already up to date.
/content
Successfully synced: Power_BI_Spark_Labs


In [ ]:
# install datamodel_code_generator if not present
!pip install datamodel-code-generator

## 1. Create Models from Schemas (with Caching)

In [215]:
import requests
import json
import re
import os
from urllib.parse import urljoin
from datamodel_code_generator import InputFileType, generate

# Define Official Fabric Schema URLs
urls = {
    "Report": "https://developer.microsoft.com/json-schemas/fabric/item/report/definition/report/3.0.0/schema.json",
    "Page": "https://developer.microsoft.com/json-schemas/fabric/item/report/definition/page/2.0.0/schema.json",
    "VisualContainer": "https://developer.microsoft.com/json-schemas/fabric/item/report/definition/visualContainer/2.3.0/schema.json",
    "Bookmark": "https://developer.microsoft.com/json-schemas/fabric/item/report/definition/bookmark/1.4.0/schema.json",
    "VersionMetadata": "https://developer.microsoft.com/json-schemas/fabric/item/report/definition/versionMetadata/1.0.0/schema.json",
    "PagesMetadata": "https://developer.microsoft.com/json-schemas/fabric/item/report/definition/pagesMetadata/1.0.0/schema.json"
}

CACHE_DIR = f"{MACH3_ROOT}/schemas_cache"
os.makedirs(CACHE_DIR, exist_ok=True)
remote_cache = {}

def get_schema(url):
    """Fetches schema from local cache or remote URL."""
    filename = url.split('/')[-1]
    if not filename.endswith('.json'): filename += '.json'
    # Simple hash-like name if URL is complex
    local_path = os.path.join(CACHE_DIR, filename if len(filename) < 50 else str(hash(url)) + ".json")

    if os.path.exists(local_path):
        with open(local_path, 'r') as f: return json.load(f)

    res = requests.get(url)
    res.raise_for_status()
    data = res.json()
    with open(local_path, 'w') as f: json.dump(data, f)
    return data

def resolve_type_from_ref(base_url, ref_path, current_schema):
    if ref_path.startswith('#'):
        parts = ref_path.strip('#/').split('/')
        curr = current_schema
        for p in parts: curr = curr.get(p, {})
        return curr.get('type', 'object')
    target_url = urljoin(base_url, ref_path.split('#')[0])
    schema = get_schema(target_url)
    if '#' in ref_path:
        parts = ref_path.split('#')[1].strip('/').split('/')
        for p in parts: schema = schema.get(p, {})
    return schema.get('type', 'object')

def clean_schema(obj, base_url, root_schema):
    if isinstance(obj, dict):
        if "description" in obj and "schema to use for an item" in obj["description"]: return {"type": "string"}
        if "$ref" in obj and isinstance(obj["$ref"], str):
            actual_type = resolve_type_from_ref(base_url, obj["$ref"], root_schema)
            return {"anyOf": [{"type": actual_type}, {"type": "null"}], "default": None}
        return {k: clean_schema(v, base_url, root_schema) for k, v in obj.items()}
    return [clean_schema(i, base_url, root_schema) for i in obj] if isinstance(obj, list) else obj

final_code = ["from __future__ import annotations", "from typing import Literal, Any, Union, List, Optional, Dict", "from pydantic import BaseModel, ConfigDict, Field, constr, RootModel"]
for name, url in urls.items():
    schema_obj = get_schema(url)
    cleaned = clean_schema(schema_obj, url, schema_obj)
    output = generate(json.dumps(cleaned), input_file_type=InputFileType.JsonSchema, output_model_type="pydantic_v2.BaseModel")
    block = re.sub(r'^(from __future__|from pydantic|from typing).*$', '', output, flags=re.MULTILINE)
    final_code.append(f"# --- {name} ---\n" + block.strip())

# Save to the repository path
output_path = os.path.join(MACH3_ROOT, "fabric_models.py")
with open(output_path, "w") as f: f.write("\n".join(final_code))
print(f"fabric_models.py created and schemas cached in {CACHE_DIR}")

fabric_models.py created and schemas cached in /content/Power_BI_Spark_Labs/Power BI/Automations/Power Bi Desktop/Mach3/schemas_cache


In [216]:
# import os
# import json
# import shutil

# def save_fabric_definition(report_obj, pages_list, bookmarks_list, extra_metadata, base_output_path):
#     """
#     Reconstructs the full modular Fabric definition folder structure.

#     :param report_obj: The Report model instance
#     :param pages_list: List of (Page instance, list of VisualContainer instances)
#     :param bookmarks_list: List of Bookmark instances
#     :param extra_metadata: Dict containing 'pages.json' and 'bookmarks.json' raw data
#     :param base_output_path: Target directory
#     """
#     os.makedirs(base_output_path, exist_ok=True)

#     # 1. report.json
#     with open(os.path.join(base_output_path, 'report.json'), 'w') as f:
#         f.write(report_obj.model_dump_json(by_alias=True, exclude_none=True, indent=2))

#     # 2. Pages metadata and folders
#     pages_base = os.path.join(base_output_path, 'pages')
#     os.makedirs(pages_base, exist_ok=True)
#     if 'pages.json' in extra_metadata:
#         with open(os.path.join(pages_base, 'pages.json'), 'w') as f:
#             json.dump(extra_metadata['pages.json'], f, indent=2)

#     for page, visuals in pages_list:
#         page_folder = os.path.join(pages_base, page.name)
#         os.makedirs(page_folder, exist_ok=True)
#         with open(os.path.join(page_folder, 'page.json'), 'w') as f:
#             f.write(page.model_dump_json(by_alias=True, exclude_none=True, indent=2))
#         if visuals:
#             v_base = os.path.join(page_folder, 'visuals')
#             for v in visuals:
#                 v_data = v.root
#                 v_folder = os.path.join(v_base, v_data.name)
#                 os.makedirs(v_folder, exist_ok=True)
#                 with open(os.path.join(v_folder, 'visual.json'), 'w') as f:
#                     f.write(v.model_dump_json(by_alias=True, exclude_none=True, indent=2))

#     # 3. Bookmarks
#     if bookmarks_list or 'bookmarks.json' in extra_metadata:
#         bookmarks_base = os.path.join(base_output_path, 'bookmarks')
#         os.makedirs(bookmarks_base, exist_ok=True)
#         if 'bookmarks.json' in extra_metadata:
#             with open(os.path.join(bookmarks_base, 'bookmarks.json'), 'w') as f:
#                 json.dump(extra_metadata['bookmarks.json'], f, indent=2)
#         for b in bookmarks_list:
#             # Using displayName or name for the filename; Fabric usually uses name.bookmark.json
#             b_name = getattr(b, 'name', 'unknown')
#             with open(os.path.join(bookmarks_base, f'{b_name}.bookmark.json'), 'w') as f:
#                 f.write(b.model_dump_json(by_alias=True, exclude_none=True, indent=2))

# print("save_fabric_definition  to include bookmarks and metadata files.")

In [217]:
# import os
# import json

# # Point src_path directly to the definition folder inside Mach3
# src_path = os.path.join(MACH3_ROOT, 'definition')

# pages_list = []
# bookmarks_list = []
# extra_metadata = {}

# if os.path.exists(src_path):
#     print(f"Loading definition from: {src_path}")
#     # Load Master
#     with open(os.path.join(src_path, 'report.json'), 'r') as f:
#         master_report = Report(**json.load(f))

#     # Load Pages
#     pages_dir = os.path.join(src_path, 'pages')
#     if os.path.exists(pages_dir):
#         for p_folder in os.listdir(pages_dir):
#             folder_path = os.path.join(pages_dir, p_folder)
#             if not os.path.isdir(folder_path): continue
#             page_json_path = os.path.join(folder_path, 'page.json')
#             if os.path.exists(page_json_path):
#                 with open(page_json_path, 'r') as f: page_obj = Page(**json.load(f))
#                 v_list = []
#                 v_dir = os.path.join(folder_path, 'visuals')
#                 if os.path.exists(v_dir):
#                     for v_f in os.listdir(v_dir):
#                         v_path = os.path.join(v_dir, v_f, 'visual.json')
#                         if os.path.exists(v_path):
#                             with open(v_path, 'r') as f: v_list.append(VisualContainer(**json.load(f)))
#                 pages_list.append((page_obj, v_list))

#     # Load Bookmarks
#     bookmarks_dir = os.path.join(src_path, 'bookmarks')
#     if os.path.exists(bookmarks_dir):
#         for b_file in os.listdir(bookmarks_dir):
#             if b_file.endswith('.bookmark.json'):
#                 with open(os.path.join(bookmarks_dir, b_file), 'r') as f:
#                     bookmarks_list.append(Bookmark(**json.load(f)))

#     # Instantiate Final Report object
#     report = FabricReport(master_report, pages_list, bookmarks_list)
#     report.get_summary()
# else:
#     print(f"Definition folder not found at {src_path}. Please ensure your definition folder is placed inside the Mach3 directory in your repository.")

In [218]:
# from fabric_models import Report, Page, VisualContainer, Bookmark

# class FabricPage:
#     def __init__(self, model, visuals):
#         self.model = model
#         self.visuals = visuals

#     def __getattr__(self, name):
#         # Delegate attribute access to the underlying Pydantic model
#         return getattr(self.model, name)

#     def __repr__(self):
#         return f"<FabricPage: {self.displayName} ({len(self.visuals)} visuals)>"

# class FabricReport:
#     def __init__(self, report_metadata, pages_with_visuals, bookmarks):
#         self.metadata = report_metadata
#         self.pages = [FabricPage(p, v) for p, v in pages_with_visuals]
#         self.bookmarks = bookmarks

#     def get_summary(self):
#         print(f"--- Fabric Report Master Summary ---")
#         print(f"Pages: {len(self.pages)} | Bookmarks: {len(self.bookmarks)}")
#         for page in self.pages:
#             print(f"- {page.displayName} ({len(page.visuals)} visuals)")

#     def __repr__(self):
#         return f"<FabricReport: {len(self.pages)} Pages>"

In [219]:
# # Instantiate the final report object using the loaded lists
# report_instance = FabricReport(master_report, pages_list, bookmarks_list)
# report_instance.get_summary()

## 2. Naming Conventions & Schema Validation
We can define a validator class to enforce organizational standards on our report definition.

In [220]:
# import re

# class FabricReportValidator:
#     def __init__(self, report):
#         self.report = report

#     def check_page_naming_convention(self, pattern=r"^[A-Z][a-zA-Z0-9\s]+"):
#         """Checks if all pages start with an uppercase letter."""
#         errors = []
#         for page in self.report.pages:
#             if not re.match(pattern, page.displayName):
#                 errors.append(f"Page '{page.displayName}' does not match convention.")
#         return errors

#     def check_duplicate_visual_names(self):
#         """Finds visuals with identical display names on the same page."""
#         duplicates = []
#         for page in self.report.pages:
#             seen = set()
#             for v in page.visuals:
#                 # Note: Some visuals might not have a displayName in the model depending on type
#                 v_name = getattr(v.root.visual, 'displayName', None) or v.root.name
#                 if v_name in seen:
#                     duplicates.append(f"Duplicate visual '{v_name}' on page '{page.displayName}'")
#                 seen.add(v_name)
#         return duplicates

# # Initialize and run basic checks
# validator = FabricReportValidator(report_instance)
# print("--- Naming Convention Audit ---")
# p_errors = validator.check_page_naming_convention()
# v_errors = validator.check_duplicate_visual_names()

# if not p_errors and not v_errors:
#     print("✅ All naming conventions passed!")
# else:
#     for err in p_errors + v_errors: print(f"❌ {err}")

In [221]:
# validation_rules = [
#     {
#         "id": "RULE_001",
#         "check": "Page Naming Convention",
#         "logic": "displayName must match r'^[A-Z][a-zA-Z0-9\\s]+'",
#         "scope": "Page Level"
#     },
#     {
#         "id": "RULE_002",
#         "check": "Unique Visual Names",
#         "logic": "No two visuals on the same page can share a displayName",
#         "scope": "Visual Level"
#     },
#     {
#         "id": "RULE_003",
#         "check": "Page Dimensions",
#         "logic": "height and width must be present if displayOption != 'dynamic'",
#         "scope": "Page Level"
#     },
#     {
#         "id": "RULE_004",
#         "check": "Active Page Metadata",
#         "logic": "activePageName must exist in pages.json and match a valid page name",
#         "scope": "Report Level"
#     },
#     {
#         "id": "RULE_005",
#         "check": "Engine Versioning",
#         "logic": "reportVersionAtImport must be present in baseTheme for PBI Desktop compatibility",
#         "scope": "Report Level"
#     }
# ]

# import pandas as pd
# display(pd.DataFrame(validation_rules))

In [222]:
# class FabricBPARules:
#     def __init__(self, report_instance, extra_metadata):
#         self.report = report_instance
#         self.metadata = extra_metadata

#     def run_all_checks(self):
#         results = []
#         # RULE_001 & RULE_002 logic from previous validator
#         results.extend(self._check_naming())
#         results.extend(self._check_dimensions())
#         results.extend(self._check_metadata())
#         results.extend(self._check_engine_version())

#         print("--- Best Practice Analyzer (BPA) Results ---")
#         if not results: print("✅ Report satisfies all Best Practices.")
#         for res in results: print(res)

#     def _check_naming(self):
#         errs = []
#         for page in self.report.pages:
#             if not page.displayName[0].isupper():
#                 errs.append(f"❌ [RULE_001]: Page '{page.displayName}' should start with uppercase.")
#         return errs

#     def _check_dimensions(self):
#         errs = []
#         for page in self.report.pages:
#             if page.displayOption != 'dynamic':
#                 if not hasattr(page, 'height') or not hasattr(page, 'width'):
#                     errs.append(f"❌ [RULE_003]: Page '{page.displayName}' missing static dimensions.")
#         return errs

#     def _check_metadata(self):
#         errs = []
#         p_meta = self.metadata.get('pages.json', {})
#         active_p = p_meta.get('activePageName')
#         page_names = [p.name for p in self.report.pages]
#         if active_p not in page_names:
#             errs.append(f"❌ [RULE_004]: activePageName '{active_p}' not found in report pages.")
#         return errs

#     def _check_engine_version(self):
#         errs = []
#         theme = getattr(self.report.metadata, 'themeCollection', None)
#         if not theme or 'reportVersionAtImport' not in str(theme):
#              errs.append("❌ [RULE_005]: Missing engine versioning in themeCollection.")
#         return errs

# # Run BPA on our new sample
# bpa = FabricBPARules(test_report, extra_metadata)
# bpa.run_all_checks()

### 5. Template-Based Report Generation
This logic uses a single 'Master Page' as a template and replicates its structure across a list of new chapters/pages.

In [223]:
# def create_pages_from_template(template_page, chapter_names):
#     """
#     Clones a template page and creates new pages with sequential names.
#     Filters out any visuals that have a 'query' property (data-bound visuals).
#     """
#     new_pages_list = []

#     for i, chapter in enumerate(chapter_names, start=1):
#         # 1. Clone and update the Page model
#         page_data = template_page.model.model_dump(by_alias=True)
#         page_suffix = f"{i:03d}"
#         page_data['name'] = f"Chapter_Page_{page_suffix}"
#         page_data['displayName'] = chapter

#         new_page_model = Page.model_validate(page_data)

#         # 2. Clone Visuals, but skip those with 'query' data
#         cloned_visuals = []
#         v_counter = 1
#         for v in template_page.visuals:
#             v_data = v.model_dump(by_alias=True)

#             # Check if the visual has a query linked
#             # VisualContainer typically wraps the visual object under 'visual'
#             visual_inner = v_data.get('visual', {})
#             if visual_inner and 'query' in visual_inner:
#                 continue # Skip visuals with queries

#             # Update name for remaining visuals
#             v_data['name'] = f"Visual_{v_counter:03d}_Chp_{page_suffix}"
#             cloned_visuals.append(VisualContainer.model_validate(v_data))
#             v_counter += 1

#         new_pages_list.append((new_page_model, cloned_visuals))

#     return new_pages_list

# # Re-run Example Usage:
# template = report_instance.pages[0]
# chapters = ["Executive Summary", "Market Trends", "Operational Risks", "Financial Outlook"]

# programmatic_pages = create_pages_from_template(template, chapters)

# print(f"Generated {len(programmatic_pages)} pages from template: {template.displayName}")
# for p, visuals in programmatic_pages:
#     print(f" - {p.displayName}: {len(visuals)} static visuals preserved (data visuals removed).")

# # validate the generated against schema
# validator = FabricReportValidator(FabricReport(master_report, programmatic_pages, bookmarks_list))

# print("--- Naming Convention Audit ---")
# p_errors = validator.check_page_naming_convention()
# v_errors = validator.check_duplicate_visual_names()

# if not p_errors and not v_errors:
#     print("✅ All naming conventions passed!")

In [224]:
# # 1. Prepare metadata and wrap into a FabricReport instance
# new_pages_metadata = {
#     "$schema": "https://developer.microsoft.com/json-schemas/fabric/item/report/definition/pagesMetadata/1.0.0/schema.json",
#     "pageOrder": [p.name for p, _ in programmatic_pages],
#     "activePageName": programmatic_pages[0][0].name
# }

# # Instantiate the wrapper for the new programmatic report
# programmatic_report_instance = FabricReport(master_report, programmatic_pages, [])

# # 2. Define the output path in Mach3
# programmatic_output_dir = os.path.join(MACH3_ROOT, 'programmatic_chapters_definition')

# # 3. Save the definition using the underlying metadata from our new instance
# save_fabric_definition(
#     report_obj=programmatic_report_instance.metadata,
#     pages_list=[(p.model, p.visuals) for p in programmatic_report_instance.pages],
#     bookmarks_list=programmatic_report_instance.bookmarks,
#     extra_metadata={"pages.json": new_pages_metadata},
#     base_output_path=programmatic_output_dir
# )

# print(f"Programmatic report instance saved to: {programmatic_output_dir}")

In [225]:
# def validate_visual_consistency(template_page, generated_page):
#     """
#     Compares visual properties between the template and generated page
#     to ensure positions and types were preserved.
#     """
#     print(f"--- Consistency Audit: {template_page.displayName} vs {generated_page.displayName} ---")

#     # Filter template visuals to only those we expected to keep (no query)
#     # We safely check .visual or default to {} to avoid NoneType errors
#     expected_template_visuals = [v for v in template_page.visuals if 'query' not in (v.root.visual or {})]

#     if len(expected_template_visuals) != len(generated_page.visuals):
#         print(f"❌ Count Mismatch: Template expected {len(expected_template_visuals)} vs Generated {len(generated_page.visuals)}")
#         return

#     matches = 0
#     for i, (t_v, g_v) in enumerate(zip(expected_template_visuals, generated_page.visuals)):
#         t_root = t_v.root
#         g_root = g_v.root

#         # Safely extract visual properties
#         t_visual = t_root.visual or {}
#         g_visual = g_root.visual or {}

#         # Check Position and Type
#         pos_match = t_root.position == g_root.position
#         type_match = t_visual.get('visualType') == g_visual.get('visualType')
#         name_changed = t_root.name != g_root.name

#         if pos_match and type_match and name_changed:
#             matches += 1
#         else:
#             print(f"❌ Mismatch at index {i}: Position Match: {pos_match}, Type Match: {type_match}, Name Unique: {name_changed}")

#     print(f"✅ {matches}/{len(expected_template_visuals)} visuals verified: Formatting & Positions preserved, IDs unique.")

# # Run the validation
# validate_visual_consistency(report_instance.pages[0], programmatic_report_instance.pages[0])

# **zip the folders to download**

In [226]:
# import shutil

# # Path to the definition we just created
# source_dir = os.path.join(MACH3_ROOT, 'programmatic_chapters_definition')
# zip_filename = '/content/sample_report'

# # Create a zip archive
# shutil.make_archive(zip_filename, 'zip', source_dir)

# print(f"Zip file created at: {zip_filename}.zip")
# print("You can download it from the files pane on the left.")

# UnZip the definition folder

In [227]:
# import zipfile
# import os

# # Updated to use your repository path
# repo_mach3_path = '/content/Power_BI_Spark_Labs/Power BI/Automations/Power Bi Desktop/Mach3'
# zip_path = os.path.join(repo_mach3_path, 'definition.zip')
# extract_path = 'fabric_report_definition'

# if os.path.exists(zip_path):
#     with zipfile.ZipFile(zip_path, 'r') as zip_ref:
#         zip_ref.extractall(extract_path)
#     print(f"Extracted {zip_path} to {extract_path}/")
# else:
#     # If the folder is already unzipped in the repo, we can point src_path directly there later
#     print(f"Zip file not found at {zip_path}. Checking if definition folder exists directly...")
#     if os.path.exists(os.path.join(repo_mach3_path, 'definition')):
#         print("Found unzipped 'definition' folder in repository.")
#     else:
#         print("Could not find definition at the specified repo path.")

In [228]:
def push_changes(commit_message="Update report definition in Mach3"):
    """Stages all changes, pulls remote updates, commits, and pushes."""
    import os

    original_dir = os.getcwd()
    os.chdir(REPO_DIR)

    try:
        # Configure git user
        !git config --global user.email "narendradasari725@gmail.com"
        !git config --global user.name "Narendra725"

        # Pull latest changes to avoid conflicts
        print("Syncing with remote...")
        !git pull origin main --rebase

        # Add and check for changes
        !git add .
        status = !git status --porcelain
        if status:
            !git commit -m "{commit_message}"
            !git push origin main
            print("Changes pushed successfully.")
        else:
            print("No changes to commit.")

    except Exception as e:
        print(f"An error occurred during git operations: {e}")
    finally:
        os.chdir(original_dir)

In [229]:
# from fabric_models import Report, Page, VisualContainer, Bookmark
# import os

# # 1. Create the Master Report Metadata with required versioning info
# new_report = Report.model_validate({
#     "$schema": "https://developer.microsoft.com/json-schemas/fabric/item/report/definition/report/3.0.0/schema.json",
#     "themeCollection": {
#         "baseTheme": {
#             "name": "CY24SU06",
#             "reportVersionAtImport": {
#                 "visual": "1.8.92",
#                 "report": "2.0.92",
#                 "page": "1.3.92"
#             },
#             "type": "SharedResources"
#         }
#     },
#     "settings": {
#         "useStylableVisualContainerHeader": True
#     }
# })

# # 2. Create a Sample Page with explicit dimensions
# new_page = Page.model_validate({
#     "$schema": "https://developer.microsoft.com/json-schemas/fabric/item/report/definition/page/2.0.0/schema.json",
#     "name": "SamplePage_001",
#     "displayName": "Programmatic Page",
#     "displayOption": "FitToPage",
#     "height": 720,
#     "width": 1280
# })

# # 3. Create a Sample Visual
# new_visual = VisualContainer.model_validate({
#     "$schema": "https://developer.microsoft.com/json-schemas/fabric/item/report/definition/visualContainer/2.3.0/schema.json",
#     "name": "SampleCard_01",
#     "position": {
#         "x": 50, "y": 50, "z": 100, "width": 200, "height": 100
#     },
#     "visual": {
#         "visualType": "card"
#     }
# })

# # 4. Assemble structure - Added activePageName to satisfy schema requirements
# pages_to_save = [(new_page, [new_visual])]
# extra_metadata = {
#     "pages.json": {
#         "$schema": "https://developer.microsoft.com/json-schemas/fabric/item/report/definition/pagesMetadata/1.0.0/schema.json",
#         "pageOrder": ["SamplePage_001"],
#         "activePageName": "SamplePage_001"
#     }
# }

# # 5. Execute Save
# output_dir = os.path.join(MACH3_ROOT, 'sample_report_definition')
# save_fabric_definition(
#     report_obj=new_report,
#     pages_list=pages_to_save,
#     bookmarks_list=[],
#     extra_metadata=extra_metadata,
#     base_output_path=output_dir
# )

# print(f"Successfully created corrected sample definition at: {output_dir}")

In [230]:
# import os
# import json

# # Point to our new sample definition
# test_src = os.path.join(MACH3_ROOT, 'sample_report_definition')

# print(f"--- Local Consistency Check: {test_src} ---")

# try:
#     # 1. Check report.json
#     with open(os.path.join(test_src, 'report.json'), 'r') as f:
#         report_data = json.load(f)
#         print("✅ report.json is valid JSON.")
#         # Check for the fix we just added
#         if 'reportVersionAtImport' in report_data.get('themeCollection', {}).get('baseTheme', {}):
#             print("✅ reportVersionAtImport found.")

#     # 2. Check pages metadata
#     with open(os.path.join(test_src, 'pages/pages.json'), 'r') as f:
#         p_meta = json.load(f)
#         order = p_meta.get('pageOrder', [])
#         print(f"✅ pages.json found with {len(order)} page(s) in order.")

#     # 3. Check for physical folder existence for every page in metadata
#     for p_name in order:
#         p_folder = os.path.join(test_src, 'pages', p_name)
#         if os.path.exists(p_folder):
#             print(f"✅ Folder for page '{p_name}' exists.")
#             if os.path.exists(os.path.join(p_folder, 'page.json')):
#                 print(f"  ✅ page.json for '{p_name}' exists.")
#         else:
#             print(f"❌ Folder for page '{p_name}' is MISSING!")

#     # 4. Try loading it into our high-level FabricReport class to verify models
#     # Note: We'll re-run the loading logic for this specific folder
#     print("\n--- Model Validation Test ---")
#     with open(os.path.join(test_src, 'report.json'), 'r') as f: t_master = Report(**json.load(f))
#     t_pages = []
#     for p_name in order:
#         with open(os.path.join(test_src, 'pages', p_name, 'page.json'), 'r') as f:
#             p_obj = Page(**json.load(f))
#             v_list = []
#             v_base = os.path.join(test_src, 'pages', p_name, 'visuals')
#             if os.path.exists(v_base):
#                 for v_folder in os.listdir(v_base):
#                     v_file = os.path.join(v_base, v_folder, 'visual.json')
#                     with open(v_file, 'r') as f: v_list.append(VisualContainer(**json.load(f)))
#             t_pages.append((p_obj, v_list))

#     test_report = FabricReport(t_master, t_pages, [])
#     test_report.get_summary()
#     print("\n✅ Local validation passed! The structure is consistent with the models.")

# except Exception as e:
#     print(f"❌ Validation Failed: {e}")

# **GIT PUSH**

In [235]:
# Push the programmatic sample definition to the repository
push_changes("Gemini Colab Push : Fabric Models Generation")

Syncing with remote...
From https://github.com/Narendra725/Power_BI_Spark_Labs
 * branch            main       -> FETCH_HEAD
Already up to date.
No changes to commit.


### Exporting Core Logic for Multi-Notebook Use
Since we want to keep this notebook for class definitions, we will export `FabricReport`, `FabricPage`, and `FabricBPARules` to a local module. This allows you to simply `from mach3_core import FabricReport` in your new workspace notebook.

In [232]:
# core_logic_path = os.path.join(MACH3_ROOT, 'mach3_core.py')

# # Building a complete core module content including missing functions
# core_content = """
# import os
# import json
# import re
# import pandas as pd
# from fabric_models import Report, Page, VisualContainer, Bookmark

# class FabricPage:
#     def __init__(self, model, visuals):
#         self.model = model
#         self.visuals = visuals
#     def __getattr__(self, name): return getattr(self.model, name)
#     def __repr__(self): return f"<FabricPage: {self.displayName} ({len(self.visuals)} visuals)>"

# class FabricReport:
#     def __init__(self, report_metadata, pages_with_visuals, bookmarks):
#         self.metadata = report_metadata
#         self.pages = [FabricPage(p, v) for p, v in pages_with_visuals]
#         self.bookmarks = bookmarks
#     def get_summary(self):
#         print(f"--- Fabric Report Master Summary ---")
#         print(f"Pages: {len(self.pages)} | Bookmarks: {len(self.bookmarks)}")
#         for page in self.pages:
#             print(f"- {page.displayName} ({len(page.visuals)} visuals)")

# def save_fabric_definition(report_obj, pages_list, bookmarks_list, extra_metadata, base_output_path):
#     os.makedirs(base_output_path, exist_ok=True)
#     with open(os.path.join(base_output_path, 'report.json'), 'w') as f:
#         f.write(report_obj.model_dump_json(by_alias=True, exclude_none=True, indent=2))
#     pages_base = os.path.join(base_output_path, 'pages')
#     os.makedirs(pages_base, exist_ok=True)
#     if 'pages.json' in extra_metadata:
#         with open(os.path.join(pages_base, 'pages.json'), 'w') as f:
#             json.dump(extra_metadata['pages.json'], f, indent=2)
#     for page, visuals in pages_list:
#         page_folder = os.path.join(pages_base, page.name)
#         os.makedirs(page_folder, exist_ok=True)
#         with open(os.path.join(page_folder, 'page.json'), 'w') as f:
#             f.write(page.model_dump_json(by_alias=True, exclude_none=True, indent=2))
#         if visuals:
#             v_base = os.path.join(page_folder, 'visuals')
#             for v in visuals:
#                 v_data = v.root
#                 v_folder = os.path.join(v_base, v_data.name)
#                 os.makedirs(v_folder, exist_ok=True)
#                 with open(os.path.join(v_folder, 'visual.json'), 'w') as f: f.write(v.model_dump_json(by_alias=True, exclude_none=True, indent=2))

# def create_pages_from_template(template_page, chapter_names):
#     new_pages_list = []
#     for i, chapter in enumerate(chapter_names, start=1):
#         page_data = template_page.model.model_dump(by_alias=True)
#         page_suffix = f"{i:03d}"
#         page_data['name'] = f"Chapter_Page_{page_suffix}"
#         page_data['displayName'] = chapter
#         new_page_model = Page.model_validate(page_data)
#         cloned_visuals = []
#         v_counter = 1
#         for v in template_page.visuals:
#             v_data = v.model_dump(by_alias=True)
#             visual_inner = v_data.get('visual', {})
#             if visual_inner and 'query' in visual_inner:
#                 continue
#             v_data['name'] = f"Visual_{v_counter:03d}_Chp_{page_suffix}"
#             cloned_visuals.append(VisualContainer.model_validate(v_data))
#             v_counter += 1
#         new_pages_list.append((new_page_model, cloned_visuals))
#     return new_pages_list

# def validate_visual_consistency(template_page, generated_page):
#     print(f"--- Consistency Audit: {template_page.displayName} vs {generated_page.displayName} ---")
#     expected_template_visuals = [v for v in template_page.visuals if 'query' not in (v.root.visual or {})]
#     if len(expected_template_visuals) != len(generated_page.visuals):
#         print(f"❌ Count Mismatch: Template expected {len(expected_template_visuals)} vs Generated {len(generated_page.visuals)}")
#         return
#     matches = 0
#     for i, (t_v, g_v) in enumerate(zip(expected_template_visuals, generated_page.visuals)):
#         t_root, g_root = t_v.root, g_v.root
#         t_visual, g_visual = t_root.visual or {}, g_root.visual or {}
#         pos_match = t_root.position == g_root.position
#         type_match = t_visual.get('visualType') == g_visual.get('visualType')
#         name_changed = t_root.name != g_root.name
#         if pos_match and type_match and name_changed: matches += 1
#     print(f"✅ {matches}/{len(expected_template_visuals)} visuals verified.")

# class FabricBPARules:
#     def __init__(self, report_instance, extra_metadata):
#         self.report = report_instance
#         self.metadata = extra_metadata
#     def run_all_checks(self):
#         results = []
#         for page in self.report.pages:
#             if not page.displayName[0].isupper():
#                 results.append(f"❌ [RULE_001]: Page '{page.displayName}' naming error.")
#         print("--- BPA Audit Complete ---")
#         for res in results: print(res)
# """

# with open(core_logic_path, 'w') as f:
#     f.write(core_content.strip())

# print(f"Updated Mach3 core logic exported to {core_logic_path}.")

In [233]:
# helper_logic_path = os.path.join(MACH3_ROOT, 'mach3_helpers.py')

# helper_content = """
# import os
# import shutil
# import zipfile

# def zip_definition(source_dir, output_zip_path):
#     \"\"\"Zips a Fabric definition folder for download.\"\"\"
#     if output_zip_path.endswith('.zip'):
#         output_zip_path = output_zip_path[:-4]
#     shutil.make_archive(output_zip_path, 'zip', source_dir)
#     print(f"Created: {output_zip_path}.zip")

# def push_to_github(repo_dir, commit_message=\"Update report\"):
#     \"\"\"Performs a standard add, commit, and push sequence.\"\"\"
#     original_dir = os.getcwd()
#     os.chdir(repo_dir)
#     try:
#         os.system('git config --global user.email "narendradasari725@gmail.com"')
#         os.system('git config --global user.name "Narendra725"')
#         os.system('git add .')
#         os.system(f'git commit -m "{commit_message}"')
#         os.system('git push origin main')
#         print("Push sequence complete.")
#     finally:
#         os.chdir(original_dir)
# """

# with open(helper_logic_path, 'w') as f:
#     f.write(helper_content.strip())

# print(f"Helper utilities exported to {helper_logic_path}")

In [234]:
# import json

# # Define a basic empty notebook structure
# empty_notebook = {
#     "cells": [],
#     "metadata": {},
#     "nbformat": 4,
#     "nbformat_minor": 5
# }

# # Path for the new workspace notebook
# new_nb_path = os.path.join(REPO_DIR, 'Report_Workspace.ipynb')

# with open(new_nb_path, 'w') as f:
#     json.dump(empty_notebook, f)

# print(f"Created empty notebook at: {new_nb_path}")

# # Push the new file to GitHub
# push_changes("Add Report_Workspace notebook")